<a href="https://colab.research.google.com/github/pushkershukla/InsightFace-PyTorch/blob/master/18_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/18_embedding.ipynb)

# 🟢 Easy: Embedding Layer

Implement an **embedding lookup table** from scratch.

### Signature
```python
class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int): ...
    def forward(self, indices: Tensor) -> Tensor: ...
```

### Rules
- `self.weight`: `nn.Parameter` of shape `(num_embeddings, embedding_dim)`
- Forward: index into weight matrix — `weight[indices]`
- Do NOT use `nn.Embedding`

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.weight=nn.Parameter(torch.randn(num_embeddings,embedding_dim))
    def forward(self, indices):
        return self.weight[indices]


In [7]:
# 🧪 Debug
emb = MyEmbedding(10, 4)
idx = torch.tensor([0, 3, 7])
print('Output shape:', emb(idx).shape)
print('Matches manual:', torch.equal(emb(idx)[0], emb.weight[0]))

Output shape: torch.Size([3, 4])
Matches manual: True


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('embedding')


🧪 Testing: Embedding Layer (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Weight shape (0.6ms)
  ✅ [2/4] Lookup correctness (0.7ms)
  ✅ [3/4] Batch of indices (7.9ms)
  ✅ [4/4] Gradient flow (48.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (57.5ms total)
  Progress saved. Run status() to see your dashboard.



#Notes
1. Embedding dimensions , you may need to add more
2. Learn to use nn.Parameter
3. Why should I use nn.Parameter vs torch.tensor  because
nn.Parameters gets optimized - The model will update paramters it finds through model.parameters()
, saved  in state Dictionalry , appears in model.Parameters, works with optimizers and is good for learnable wieghts
4. Remember embedding layer is a look up table that converts each input into an embedding vector , so [4,1,3] if passed through an embedding layer will be converted into an embedding vector of size (3,128) if embdding_dim is 128. It is like a lookup table
5. Torch.stack and how to use it

## Embedding Interview Questions

1. **What is an embedding and why is it used instead of one-hot encoding?**

2. **Explain how an embedding layer works internally in PyTorch. What operation is actually performed during the forward pass?**

3. **What is the shape of an embedding matrix, and how does the input shape affect the output shape?**

4. **How would you implement an embedding layer from scratch using PyTorch?**

5. **What is the difference between an embedding layer and a linear (fully connected) layer?**

6. **How are embeddings learned during training? Which parts of the embedding matrix get updated?**

7. **Why must embedding indices be of type `torch.long`? What happens if they are not?**

8. **Is embedding lookup equivalent to multiplying a one-hot vector with a weight matrix? Explain both the equivalence and the practical difference.**

9. **What are positional embeddings and why are they necessary in transformer models?**

10. **How would you analyze or detect bias in learned embeddings?**

## Embedding Interview Questions — Answers

1. **What is an embedding and why is it used instead of one-hot encoding?**  
An embedding is a dense, low-dimensional vector representation of a discrete item (e.g., a word or token). Unlike one-hot vectors, embeddings capture semantic relationships between items, are memory-efficient, and allow models to generalize better.

---

2. **Explain how an embedding layer works internally in PyTorch. What operation is actually performed during the forward pass?**  
An embedding layer performs a lookup operation: given indices, it retrieves the corresponding rows from a weight matrix. Internally, it is equivalent to `weight[indices]`, not a matrix multiplication.

---

3. **What is the shape of an embedding matrix, and how does the input shape affect the output shape?**  
The embedding matrix has shape `(vocab_size, embedding_dim)`.  
If the input has shape `(batch_size, sequence_length)`, the output will have shape `(batch_size, sequence_length, embedding_dim)`.

---

4. **How would you implement an embedding layer from scratch using PyTorch?**  
```python
import torch
import torch.nn as nn

class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))

    def forward(self, indices):
        return self.weight[indices]

## Embedding Interview Questions — Answers (5–8)

5. **What is the difference between an embedding layer and a linear (fully connected) layer?**  
An embedding layer performs a lookup (indexing rows from a matrix), while a linear layer performs a matrix multiplication (`y = Wx + b`). Embeddings take discrete indices as input, whereas linear layers take continuous vectors. Embeddings are computationally cheaper for sparse inputs.

---

6. **How are embeddings learned during training? Which parts of the embedding matrix get updated?**  
Embeddings are learned via backpropagation. During the forward pass, only specific rows are selected based on input indices. During the backward pass, **only those selected rows receive gradients and get updated**, making the updates sparse and efficient.

---

7. **Why must embedding indices be of type `torch.long`? What happens if they are not?**  
Indices must be integers because they represent positions in the embedding matrix. In PyTorch, this requires `torch.long` dtype. If a different dtype (like `float`) is used, PyTorch will raise a runtime error since indexing with non-integer values is invalid.

---

8. **Is embedding lookup equivalent to multiplying a one-hot vector with a weight matrix? Explain both the equivalence and the practical difference.**  
Yes, mathematically they are equivalent: multiplying a one-hot vector with a weight matrix selects the corresponding row. However, embedding lookup is much more efficient because it avoids constructing large sparse one-hot vectors and performing unnecessary multiplications.